# Notebook 2 : Création du dataset SFT multi-tâches

Ce notebook transforme le dataset agrégé en dataset de supervised fine-tuning, SFT.

--> les données sont converties en exemples sous forme de prompts.

Chaque exemple suit une structure conversationnelle :

- system prompt 
- user prompt 
- assistant answer

L’objectif est de fine-tuner un seul modèle de langage en mode multi-tâches.

**Point important**

Chaque exemple SFT correspond à une seule tâche.

- Un exemple Task A contient uniquement le prompt de Task A (classification en 3 catégories).
- Un exemple Task B contient uniquement le prompt de Task B (génétation de l'argument implicite).
- Un exemple Task C contient uniquement le prompt de Task C (task A + task B)

Cependant, tous ces exemples sont regroupés dans un même fichier afin d’entraîner un seul modèle en mode multi-tâches.

**Fichiers produits**

- data/sft/train_sft.jsonl
- data/sft/validation_sft.jsonl
- data/sft/test_sft.jsonl

Ces fichiers sont ensuite utilisés pour le fine-tuning du modèle.

### Chargement 

In [1]:
import json
import random
from pathlib import Path
from collections import Counter
from typing import Any, Dict, List, Optional

import pandas as pd

In [2]:
PROCESSED_DIR = Path("data/processed")
SFT_DIR = Path("data/sft")
SFT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = PROCESSED_DIR / "train_aggregated.jsonl"
VALIDATION_PATH = PROCESSED_DIR / "validation_aggregated.jsonl"
TEST_PATH = PROCESSED_DIR / "test_aggregated.jsonl"

SEED = 42
random.seed(SEED)

### Chargement de dataset aggregé 

In [3]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    return records

In [4]:
train_records = read_jsonl(TRAIN_PATH)
validation_records = read_jsonl(VALIDATION_PATH)
test_records = read_jsonl(TEST_PATH)

print("Train:", len(train_records))
print("Validation:", len(validation_records))
print("Test:", len(test_records))

Train: 1066
Validation: 133
Test: 134


In [5]:
train_records[0]

{'id': '864',
 'tweet_text': 'What part of do you vaccine takers not understand. STOP TAKING THAT SHIT. The KILLER VACCINES. JUST STOP. THEY ARE FAKE & Will eventually KILL YOU',
 'n_annotators': 5,
 'annotations_normalized': [{'annotator_id': 0,
   'label': 'premise',
   'implicit_text': 'If something will eventually kill you, you should not take it'},
  {'annotator_id': 1, 'label': 'none', 'implicit_text': ''},
  {'annotator_id': 2, 'label': 'none', 'implicit_text': ''},
  {'annotator_id': 3, 'label': 'none', 'implicit_text': ''},
  {'annotator_id': 4, 'label': 'none', 'implicit_text': ''}],
 'label_counts': {'none': 4, 'premise': 1, 'claim': 0},
 'vote_distribution': {'none': 0.8, 'premise': 0.2, 'claim': 0.0},
 'majority_label': 'none',
 'implicit_status': 'no',
 'confidence': 0.8,
 'uncertainty_level': 'low',
 'has_label_tie': False,
 'tied_labels': ['none'],
 'valid_implicit_texts': ['If something will eventually kill you, you should not take it'],
 'majority_implicit_texts': [],

### Check for ambiguous 

In [ ]:
required_keys = [
    "tweet_text",
    "majority_label",
    "implicit_status",
    "vote_distribution",
    "uncertainty_level",
    "annotations_normalized",
    "valid_implicit_texts",
    "majority_implicit_texts",
    "canonical_implicit_text",
    "usable_for_classification",
    "usable_for_generation",
]

missing_keys = []

for key in required_keys:
    if key not in train_records[0]:
        missing_keys.append(key)

if missing_keys:
    print("Missing keys:", missing_keys)
    print("Faut revenir au dataset agrégé.")
else:
    print("All required keys are present.")

All required keys are present.


In [7]:
pd.Series([r["majority_label"] for r in train_records]).value_counts()

none         708
premise      294
claim         39
ambiguous     25
Name: count, dtype: int64

### Main parametres SFT

In [8]:
VALID_LABELS = ["none", "premise", "claim"]
EXTENDED_LABELS = ["none", "premise", "claim", "ambiguous"]

SYSTEM_PROMPT = """You are an expert annotator of enthymemes and argumentative structures.

Your task is to analyze short social media texts.
You must identify whether the text contains missing implicit argumentation.
If an implicit argument exists, classify it as either:
- implicit premise: a missing supporting reason or assumption
- implicit claim: a missing conclusion inferred from explicit premises

Return only valid JSON.
Do not add explanations outside JSON.
"""

### Fonctions

In [9]:
def dumps_json(obj: Dict[str, Any]) -> str:
    """
    Красиво сериализует JSON без ASCII-escaping.
    """
    return json.dumps(obj, ensure_ascii=False, indent=2)

In [ ]:
def make_message_record(
    system_prompt: str,
    user_prompt: str,
    assistant_output: Dict[str, Any],
    metadata: Dict[str, Any],
) -> Dict[str, Any]:
    """

    Crées un enregistrement de message SFT à partir des prompts et de la sortie de l'assistant.

    Nous sauvegardons:
    - messages: pour le notebook d'entraînement, où nous appliquerons tokenizer.apply_chat_template
    - text_preview: pour un aperçu pratique
    - metadata: task_type, id, split, etc.
    """

    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
        {
            "role": "assistant",
            "content": dumps_json(assistant_output),
        },
    ]

    text_preview = (
        f"<system>\n{system_prompt}\n</system>\n\n"
        f"<user>\n{user_prompt}\n</user>\n\n"
        f"<assistant>\n{dumps_json(assistant_output)}\n</assistant>"
    )

    return {
        "messages": messages,
        "text_preview": text_preview,
        "metadata": metadata,
    }

### Controlled argumentative analysis
 
L’objectif n’est pas de produire une longue explication, mais de générer une justification courte, stable et contrôlée.

Cette analyse est construite automatiquement à partir des informations déjà présentes dans le dataset agrégé :

- le label majoritaire ;
- la distribution des votes entre annotateurs ;
- le niveau d’incertitude ;
- les propositions implicites disponibles.

In [ ]:
def make_argument_analysis(record: Dict[str, Any], label: Optional[str] = None, implicit_text: str = "") -> str:
    """
Crée une courte analyse argumentative structurée.

Cette analyse n'est pas une chaîne de pensée libre et détaillée.
Il s'agit d'une justification contrôlée, construite automatiquement
à partir du label majoritaire, du désaccord entre annotateurs
et des propositions implicites disponibles.

L'objectif est de fournir au modèle une explication courte et stable,
sans l'entraîner à produire de longs raisonnements non contrôlés.
"""

    majority_label = record["majority_label"]
    vote_distribution = record["vote_distribution"]
    uncertainty = record["uncertainty_level"]

    if label is None:
        label = majority_label

    if label == "none":
        return (
            "The annotators mostly judged that the tweet does not require an additional hidden premise "
            "or hidden claim to complete its argumentative structure."
        )

    if label == "premise":
        if implicit_text:
            return (
                "The tweet contains an explicit evaluative or argumentative statement. "
                "The missing proposition functions as a supporting premise: "
                f"{implicit_text}"
            )
        return (
            "The tweet contains an explicit claim, but part of the supporting reasoning is left implicit. "
            "The missing proposition is best treated as an implicit premise."
        )

    if label == "claim":
        if implicit_text:
            return (
                "The tweet provides explicit considerations that point toward a conclusion. "
                "The missing proposition functions as the implicit claim: "
                f"{implicit_text}"
            )
        return (
            "The tweet provides explicit premises or reasons, while the conclusion is left implicit. "
            "The missing proposition is best treated as an implicit claim."
        )

    if label == "ambiguous":
        tied_labels = record.get("tied_labels", [])
        return (
            "The annotations show disagreement about the argumentative role of the missing proposition. "
            f"The vote distribution is {vote_distribution}, with uncertainty level '{uncertainty}'. "
            f"The tied or competing labels are: {tied_labels}."
        )

    return (
        "The argumentative structure is uncertain, and the annotations do not provide a stable single label."
    )

### Task A: classification + uncertainty

tweet → implicit_status + implicit_type + vote_distribution + uncertainty

In [12]:
def build_task_a_user_prompt(record: Dict[str, Any]) -> str:
    tweet = record["tweet_text"]

    return f"""Task A: Analyze the enthymeme and classify the missing implicit argumentation.

You must decide whether the tweet contains missing implicit argumentation.
If yes, classify the missing argument as:
- premise
- claim

If there is no missing implicit argumentation, use:
- none

If the historical annotations are strongly divided, mark the case as ambiguous.

Tweet:
{tweet}

Return JSON with the following fields:
- argument_analysis
- implicit_status: yes / no / ambiguous
- implicit_type: none / premise / claim / ambiguous
- vote_distribution
- confidence
- uncertainty_level
"""

In [13]:
def build_task_a_output(record: Dict[str, Any]) -> Dict[str, Any]:
    majority_label = record["majority_label"]

    if majority_label == "ambiguous":
        implicit_status = "ambiguous"
        implicit_type = "ambiguous"
        analysis_label = "ambiguous"
    elif majority_label == "none":
        implicit_status = "no"
        implicit_type = "none"
        analysis_label = "none"
    else:
        implicit_status = "yes"
        implicit_type = majority_label
        analysis_label = majority_label

    return {
        "argument_analysis": make_argument_analysis(
            record,
            label=analysis_label,
            implicit_text=record.get("canonical_implicit_text", ""),
        ),
        "implicit_status": implicit_status,
        "implicit_type": implicit_type,
        "vote_distribution": record["vote_distribution"],
        "confidence": record["confidence"],
        "uncertainty_level": record["uncertainty_level"],
    }

### Task B: generation of missing argument

tweet + implicit_type + uncertainty → implicit_argument

On utilise toutes les variants données par les annotateurd et pas seulement les canonicals 

In [14]:
def build_task_b_user_prompt(
    record: Dict[str, Any],
    implicit_type: str,
) -> str:
    tweet = record["tweet_text"]

    vote_distribution = record["vote_distribution"]
    uncertainty_level = record["uncertainty_level"]

    return f"""Task B: Generate the missing implicit argument.

The implicit argument type is given.
Generate only the missing proposition that completes the enthymeme.

Tweet:
{tweet}

Implicit argument type:
{implicit_type}

Historical vote distribution:
none: {vote_distribution.get("none", 0.0)}
premise: {vote_distribution.get("premise", 0.0)}
claim: {vote_distribution.get("claim", 0.0)}

Historical uncertainty level:
{uncertainty_level}

Return JSON with the following fields:
- implicit_type
- implicit_argument
"""

In [15]:
def build_task_b_output(
    implicit_type: str,
    implicit_text: str,
) -> Dict[str, Any]:
    return {
        "implicit_type": implicit_type,
        "implicit_argument": implicit_text,
    }

### Task C: full pipeline

tweet → analysis + classification + uncertainty + generation

In [17]:
def build_task_c_user_prompt(record: Dict[str, Any]) -> str:
    tweet = record["tweet_text"]

    return f"""Task C: Analyze the enthymeme, classify the missing implicit argumentation, and reconstruct the missing proposition.

You must:
1. Analyze the argumentative structure.
2. Decide whether there is missing implicit argumentation.
3. Classify it as none, premise, or claim.
4. If the type is premise or claim, generate the missing proposition.
5. Estimate uncertainty using the pattern of historical annotations.

Tweet:
{tweet}

Return JSON with the following fields:
- argument_analysis
- implicit_status: yes / no
- implicit_type: none / premise / claim
- vote_distribution
- confidence
- uncertainty_level
- implicit_argument
"""

In [ ]:
def build_task_c_output(record: Dict[str, Any]) -> Dict[str, Any]:
    majority_label = record["majority_label"]

    if majority_label == "none":
        implicit_status = "no"
        implicit_type = "none"
        implicit_argument = ""
    else:
        implicit_status = "yes"
        implicit_type = majority_label
        implicit_argument = record.get("canonical_implicit_text", "")

    return {
        "argument_analysis": make_argument_analysis(
            record,
            label=implicit_type,
            implicit_text=implicit_argument,
        ),
        "implicit_status": implicit_status,
        "implicit_type": implicit_type,
        "vote_distribution": record["vote_distribution"],
        "confidence": record["confidence"],
        "uncertainty_level": record["uncertainty_level"],
        "implicit_argument": implicit_argument,
    }

### Création de SFT exemples pour 1 tweet 

In [19]:
def create_sft_examples_for_split(
    records: List[Dict[str, Any]],
    split_name: str,
    include_task_a: bool = True,
    include_task_b: bool = True,
    include_task_c: bool = True,
    include_ambiguous_in_task_a: bool = True,
    include_ambiguous_in_task_c: bool = False,
) -> List[Dict[str, Any]]:
    """
    Создаёт multi-task SFT examples.

    Task A:
        classification + uncertainty.
        Можно включать ambiguous, чтобы модель училась uncertainty.

    Task B:
        generation.
        Используем все non-empty implicit_text от аннотаторов.

    Task C:
        full pipeline.
        Лучше исключить ambiguous для первой версии.
    """

    sft_examples = []

    for record in records:
        tweet_id = record["id"]
        majority_label = record["majority_label"]

        # -----------------------
        # Task A: classification
        # -----------------------
        if include_task_a:
            if majority_label != "ambiguous" or include_ambiguous_in_task_a:
                user_prompt = build_task_a_user_prompt(record)
                assistant_output = build_task_a_output(record)

                metadata = {
                    "tweet_id": tweet_id,
                    "split": split_name,
                    "task_type": "task_a_classification_uncertainty",
                    "majority_label": majority_label,
                    "uncertainty_level": record["uncertainty_level"],
                    "usable_for_hard_classification_eval": record.get("usable_for_classification", False),
                }

                sft_examples.append(
                    make_message_record(
                        system_prompt=SYSTEM_PROMPT,
                        user_prompt=user_prompt,
                        assistant_output=assistant_output,
                        metadata=metadata,
                    )
                )

        # -----------------------
        # Task B: generation
        # -----------------------
        if include_task_b:
            for ann in record.get("annotations_normalized", []):
                ann_label = ann.get("label", "none")
                ann_text = ann.get("implicit_text", "")

                if ann_label in ["premise", "claim"] and ann_text:
                    user_prompt = build_task_b_user_prompt(
                        record=record,
                        implicit_type=ann_label,
                    )

                    assistant_output = build_task_b_output(
                        implicit_type=ann_label,
                        implicit_text=ann_text,
                    )

                    metadata = {
                        "tweet_id": tweet_id,
                        "split": split_name,
                        "task_type": "task_b_generation",
                        "annotation_label": ann_label,
                        "majority_label": majority_label,
                        "uncertainty_level": record["uncertainty_level"],
                        "annotator_id": ann.get("annotator_id"),
                    }

                    sft_examples.append(
                        make_message_record(
                            system_prompt=SYSTEM_PROMPT,
                            user_prompt=user_prompt,
                            assistant_output=assistant_output,
                            metadata=metadata,
                        )
                    )

        # -----------------------
        # Task C: full pipeline
        # -----------------------
        if include_task_c:
            can_use_for_task_c = majority_label in ["none", "premise", "claim"]

            if majority_label == "ambiguous" and include_ambiguous_in_task_c:
                can_use_for_task_c = True

            if can_use_for_task_c and majority_label != "ambiguous":
                if majority_label == "none" or record.get("canonical_implicit_text", ""):
                    user_prompt = build_task_c_user_prompt(record)
                    assistant_output = build_task_c_output(record)

                    metadata = {
                        "tweet_id": tweet_id,
                        "split": split_name,
                        "task_type": "task_c_full_pipeline",
                        "majority_label": majority_label,
                        "uncertainty_level": record["uncertainty_level"],
                        "usable_for_full_pipeline_eval": record.get("usable_for_classification", False),
                    }

                    sft_examples.append(
                        make_message_record(
                            system_prompt=SYSTEM_PROMPT,
                            user_prompt=user_prompt,
                            assistant_output=assistant_output,
                            metadata=metadata,
                        )
                    )

    random.shuffle(sft_examples)
    return sft_examples

### Création train / validation / test SFT datasets

In [20]:
train_sft = create_sft_examples_for_split(
    train_records,
    split_name="train",
    include_task_a=True,
    include_task_b=True,
    include_task_c=True,
    include_ambiguous_in_task_a=True,
    include_ambiguous_in_task_c=False,
)

validation_sft = create_sft_examples_for_split(
    validation_records,
    split_name="validation",
    include_task_a=True,
    include_task_b=True,
    include_task_c=True,
    include_ambiguous_in_task_a=True,
    include_ambiguous_in_task_c=False,
)

test_sft = create_sft_examples_for_split(
    test_records,
    split_name="test",
    include_task_a=True,
    include_task_b=True,
    include_task_c=True,
    include_ambiguous_in_task_a=True,
    include_ambiguous_in_task_c=False,
)

print("Train SFT examples:", len(train_sft))
print("Validation SFT examples:", len(validation_sft))
print("Test SFT examples:", len(test_sft))

Train SFT examples: 3932
Validation SFT examples: 502
Test SFT examples: 498


### Stats

In [21]:
def sft_stats(sft_examples: List[Dict[str, Any]], name: str) -> pd.DataFrame:
    rows = []

    for ex in sft_examples:
        metadata = ex["metadata"]
        rows.append(metadata)

    df = pd.DataFrame(rows)

    print(f"\n===== {name.upper()} =====")
    print("Total examples:", len(df))

    print("\nTask type distribution:")
    display(df["task_type"].value_counts())

    print("\nMajority label distribution:")
    display(df["majority_label"].value_counts())

    print("\nUncertainty distribution:")
    display(df["uncertainty_level"].value_counts())

    return df

In [22]:
train_sft_df = sft_stats(train_sft, "train")
validation_sft_df = sft_stats(validation_sft, "validation")
test_sft_df = sft_stats(test_sft, "test")


===== TRAIN =====
Total examples: 3932

Task type distribution:


task_type
task_b_generation                    1825
task_a_classification_uncertainty    1066
task_c_full_pipeline                 1041
Name: count, dtype: int64


Majority label distribution:


majority_label
premise      1901
none         1682
claim         242
ambiguous     107
Name: count, dtype: int64


Uncertainty distribution:


...

task_type
task_b_generation                    235
task_a_classification_uncertainty    134
task_c_full_pipeline                 129
Name: count, dtype: int64


Majority label distribution:


majority_label
premise      236
none         212
claim         30
ambiguous     20
Name: count, dtype: int64


Uncertainty distribution:


uncertainty_level
low       411
medium     67
high       20
Name: count, dtype: int64

### Check of one SFT exemple 

In [23]:
example = train_sft[0]

print("METADATA:")
print(dumps_json(example["metadata"]))

print("\nTEXT PREVIEW:")
print(example["text_preview"])

METADATA:
{
  "tweet_id": "663",
  "split": "train",
  "task_type": "task_a_classification_uncertainty",
  "majority_label": "premise",
  "uncertainty_level": "low",
  "usable_for_hard_classification_eval": true
}

TEXT PREVIEW:
<system>
You are an expert annotator of enthymemes and argumentative structures.

Your task is to analyze short social media texts.
You must identify whether the text contains missing implicit argumentation.
If an implicit argument exists, classify it as either:
- implicit premise: a missing supporting reason or assumption
- implicit claim: a missing conclusion inferred from explicit premises

Return only valid JSON.
Do not add explanations outside JSON.

</system>

<user>
Task A: Analyze the enthymeme and classify the missing implicit argumentation.

You must decide whether the tweet contains missing implicit argumentation.
If yes, classify the missing argument as:
- premise
- claim

If there is no missing implicit argumentation, use:
- none

If the historical

In [24]:
def show_example_by_task(sft_examples: List[Dict[str, Any]], task_type: str):
    for ex in sft_examples:
        if ex["metadata"]["task_type"] == task_type:
            print("METADATA:")
            print(dumps_json(ex["metadata"]))
            print("\nTEXT PREVIEW:")
            print(ex["text_preview"])
            return

    print(f"No example found for task_type={task_type}")

In [25]:
show_example_by_task(train_sft, "task_a_classification_uncertainty")

METADATA:
{
  "tweet_id": "663",
  "split": "train",
  "task_type": "task_a_classification_uncertainty",
  "majority_label": "premise",
  "uncertainty_level": "low",
  "usable_for_hard_classification_eval": true
}

TEXT PREVIEW:
<system>
You are an expert annotator of enthymemes and argumentative structures.

Your task is to analyze short social media texts.
You must identify whether the text contains missing implicit argumentation.
If an implicit argument exists, classify it as either:
- implicit premise: a missing supporting reason or assumption
- implicit claim: a missing conclusion inferred from explicit premises

Return only valid JSON.
Do not add explanations outside JSON.

</system>

<user>
Task A: Analyze the enthymeme and classify the missing implicit argumentation.

You must decide whether the tweet contains missing implicit argumentation.
If yes, classify the missing argument as:
- premise
- claim

If there is no missing implicit argumentation, use:
- none

If the historical

In [26]:
show_example_by_task(train_sft, "task_b_generation")

METADATA:
{
  "tweet_id": "1793",
  "split": "train",
  "task_type": "task_b_generation",
  "annotation_label": "premise",
  "majority_label": "none",
  "uncertainty_level": "medium",
  "annotator_id": 4
}

TEXT PREVIEW:
<system>
You are an expert annotator of enthymemes and argumentative structures.

Your task is to analyze short social media texts.
You must identify whether the text contains missing implicit argumentation.
If an implicit argument exists, classify it as either:
- implicit premise: a missing supporting reason or assumption
- implicit claim: a missing conclusion inferred from explicit premises

Return only valid JSON.
Do not add explanations outside JSON.

</system>

<user>
Task B: Generate the missing implicit argument.

The implicit argument type is given.
Generate only the missing proposition that completes the enthymeme.

Tweet:
The insistence on mandatory vaccines over bodily autonomy objections does not play well now regarding abortion. My body my choice rings hol

In [27]:
show_example_by_task(train_sft, "task_c_full_pipeline")

METADATA:
{
  "tweet_id": "95",
  "split": "train",
  "task_type": "task_c_full_pipeline",
  "majority_label": "none",
  "uncertainty_level": "low",
  "usable_for_full_pipeline_eval": true
}

TEXT PREVIEW:
<system>
You are an expert annotator of enthymemes and argumentative structures.

Your task is to analyze short social media texts.
You must identify whether the text contains missing implicit argumentation.
If an implicit argument exists, classify it as either:
- implicit premise: a missing supporting reason or assumption
- implicit claim: a missing conclusion inferred from explicit premises

Return only valid JSON.
Do not add explanations outside JSON.

</system>

<user>
Task C: Analyze the enthymeme, classify the missing implicit argumentation, and reconstruct the missing proposition.

You must:
1. Analyze the argumentative structure.
2. Decide whether there is missing implicit argumentation.
3. Classify it as none, premise, or claim.
4. If the type is premise or claim, generate t

### Sauvegarder SFT dataset

In [28]:
def write_jsonl(records: List[Dict[str, Any]], path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [29]:
write_jsonl(train_sft, SFT_DIR / "train_sft.jsonl")
write_jsonl(validation_sft, SFT_DIR / "validation_sft.jsonl")
write_jsonl(test_sft, SFT_DIR / "test_sft.jsonl")

print("Saved:")
print(SFT_DIR / "train_sft.jsonl")
print(SFT_DIR / "validation_sft.jsonl")
print(SFT_DIR / "test_sft.jsonl")

Saved:
data/sft/train_sft.jsonl
data/sft/validation_sft.jsonl
data/sft/test_sft.jsonl
